# 02 — PCA Signal
**PCA Macro Slowdown Indicator**
Siddhanth Yadav · Kavin Dhanasekar · Sudhan Adithya

This notebook covers:
- Full transform → standardize → sign-align pipeline
- PCA fit and PC1 extraction
- Loadings table and interpretation
- Scree plot (variance explained)
- PC1 time series with NBER recession shading

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
load_dotenv('../.env')

import config
from data.fetch_data import get_fred_client, fetch_all_macro, fetch_all_financial, fetch_reference_series
from data.align import align_macro_to_monthly, align_financial_to_monthly, merge_panel, trim_to_overlap
from data.transform import apply_transformations, drop_leading_nans
from data.standardize import standardize_panel, apply_sign_alignment, prepare_pca_matrix
from pca.build_indicator import run_pca, extract_pc1, normalize_pc1_sign, loadings_table, variance_explained
from pca.regime import smooth_pc1, nber_recession_periods
from viz.charts import plot_pc1_signal, plot_loadings, plot_scree, shade_recessions

In [ ]:
# --- Load and prepare data ---
fred   = get_fred_client()
m_raw  = fetch_all_macro(fred)
f_raw  = fetch_all_financial(fred)
r_raw  = fetch_reference_series(fred)

panel  = merge_panel(align_macro_to_monthly(m_raw), align_financial_to_monthly(f_raw), r_raw)
panel  = trim_to_overlap(panel)

panel_t   = apply_transformations(panel)
pca_cols  = [c for c in config.MACRO_PCA_COLS if c in panel_t.columns]
fin_cols  = [c for c in config.FINANCIAL_COLS  if c in panel_t.columns]
all_cols  = pca_cols + fin_cols
panel_t   = drop_leading_nans(panel_t, cols=pca_cols)

panel_std = standardize_panel(panel_t, cols=all_cols)
panel_std = apply_sign_alignment(panel_std, cols=all_cols)

Z = prepare_pca_matrix(panel_std, pca_cols=pca_cols)
print('Z matrix shape:', Z.shape)

In [ ]:
# --- Run PCA ---
pca, scores  = run_pca(Z)
pc1_raw      = extract_pc1(pca, scores, Z)
nber         = panel_std.get('nber_recession')
pc1          = normalize_pc1_sign(pc1_raw, recession_mask=nber)
pc1_smooth   = smooth_pc1(pc1)
print(pc1.describe())

In [ ]:
# --- Loadings table ---
ltbl = loadings_table(pca, list(Z.columns))
print('\nPC1 Loadings:')
ltbl[['PC1', 'PC2']].round(3)

In [ ]:
# --- Loadings bar chart ---
fig, ax = plot_loadings(ltbl, component='PC1')
plt.show()

In [ ]:
# --- Scree plot ---
vtbl = variance_explained(pca)
print(vtbl)
fig, ax = plot_scree(vtbl)
plt.show()

In [ ]:
# --- PC1 signal chart with recession shading ---
from pca.regime import nber_recession_periods
rec_periods = nber_recession_periods(nber) if nber is not None else []
fig, ax = plot_pc1_signal(pc1, pc1_smooth, recession_periods=rec_periods)
plt.show()